## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/indore/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/indore/


In [6]:
##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Jaipur'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Bangalore, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag_v1.csv', index=False)

In [7]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag_v1.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(250719, 2)


,customer_id,usecase_tag
0,5d5a6cb3bde10e0d544743ba,Unknown
1,638cea9386ff453bd97f9d43,Unknown


### Active customer (RR-customers)

In [8]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/indore_customers_v1.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list_set', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(130595, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,5e3b93c0d6d3936cd767fbdb,MALE,255.0,30.0,6.0,3.0,2.0,PHH,14,HOOK,HIGH_INCOME,ONLY_AUTO,LONG_DISTANCE,NPS,2.0,c. medium rpc
1,612f6a009db16b0477ee6780,MALE,949.0,NaN,2.0,2.0,2.0,PHH,50,SUSTENANCE,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,PS,2.0,c. medium rpc


In [9]:
# df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
#                                         how='left', on=['customer_id'])
df_bangalore_active_customer = df_bangalore_active_customer

### customer installed apps

In [10]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped_v1.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(130491, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f59b0ffc283676f94b,"['zomato', 'bookmyshow', 'kite by zerodha', 'u...",20,"['Tools', 'Educational', 'Delivery_Food', 'OTT...",10,"['Educational', 'Ride haling', 'Finance_Invest...",4,0,1,0,0,1,1,1
1,573f28fe9b0ffc28367719f1,"['zoom', 'linkedin', 'microsoft teams', 'netfl...",25,"['Tools', 'Social', 'News', 'OTT', 'Delivery_G...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1


### Customer app & cat explode mapping

In [11]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping_v1.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
#                                                        'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l2
0,5e3b93c0d6d3936cd767fbdb,Rest


In [12]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

130491

In [13]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Tools' 'Finance_Transactions' 'Delivery_Food' 'OTT' 'Social' 'Office'
 'Finance_Investment' 'News' 'Travel_Ridehailing' 'Educational' 'Gaming'
 'Streaming_Music' 'Entertainment' 'Travel_Bookings' 'Ecommerce'
 'Delivery_Grocery' 'Wearable' 'Dating' 'Finance_News' 'Fantasy_Sports'
 'Driver_App' 'Health' 'Devotional']


In [14]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Office' 'Finance_Investment' 'Ride haling' 'Educational' 'Gaming'
 'Driver_App']


In [15]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [16]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,40179,30.79
1,Ride haling,uber,23002,17.63
2,Ride haling,jugnoo,11885,9.11
3,Ride haling,in drive,2035,1.56
4,Ride haling,blusmart,224,0.17
5,Ride haling,namma yatri,156,0.12
6,Ride haling,quick ride,78,0.06
7,Ride haling,driveu driver app,7,0.01
8,Ride haling,quickride,1,0.00
9,Office,zoom,27591,21.14


### Merge raw data

In [17]:
df_bangalore_active_customer

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,5e3b93c0d6d3936cd767fbdb,MALE,255.0,30.0,6.0,3.0,2.0,PHH,14,HOOK,HIGH_INCOME,ONLY_AUTO,LONG_DISTANCE,NPS,2.0,c. medium rpc
1,612f6a009db16b0477ee6780,MALE,949.0,NaN,2.0,2.0,2.0,PHH,50,SUSTENANCE,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,PS,2.0,c. medium rpc
2,61ed8519ebef82b20323a869,FEMALE,805.0,NaN,34.0,22.0,19.0,PHH,157,COMMITTED,HIGH_INCOME,ONLY_LINK,SHORT_DISTANCE,NPS,19.0,d. high rpc
3,5db429f16699c34f1936f0dd,MALE,1625.0,26.0,6.0,4.0,2.0,PHH,56,CHURN_OTB,HIGH_INCOME,ONLY_LINK,SHORT_DISTANCE,UNKNOWN,2.0,c. medium rpc
4,5ca6f25b54bc7263ff283c6a,MALE,1829.0,28.0,5.0,3.0,2.0,PHH,36,HOOK,HIGH_INCOME,ONLY_AUTO,SHORT_DISTANCE,UNKNOWN,2.0,c. medium rpc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130590,5c0e442c7bf66a68e79a140a,FEMALE,1945.0,30.0,5.0,1.0,1.0,PHH,13,HOOK,HIGH_INCOME,ONLY_LINK,UNKNOWN,NPS,1.0,b. low rpc
130591,5ccdad9f6e78ee6df44d79c5,MALE,1800.0,29.0,1.0,1.0,0.0,PHH,54,DORMANT,UNKNOWN,ONLY_LINK,UNKNOWN,UNKNOWN,NaN,a. zero rpc
130592,65977bd0a531053370381f8d,MALE,93.0,NaN,9.0,1.0,0.0,HH,4,HANDHOLDING,MEDIUM_INCOME,ONLY_AUTO,LONG_DISTANCE,PS,NaN,a. zero rpc
130593,5d5f3ab4d0286d106d5e87f0,FEMALE,1689.0,25.0,1.0,1.0,1.0,PHH,130,CHURN_OTB,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,PS,1.0,b. low rpc


In [18]:
df_bangalore_active_customer[df_bangalore_active_customer['customer_id'].isin(['65e4c108d4f28464de89d5b0'])]

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc


In [19]:
df_customer_mapped

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f59b0ffc283676f94b,"['zomato', 'bookmyshow', 'kite by zerodha', 'u...",20,"['Tools', 'Educational', 'Delivery_Food', 'OTT...",10,"['Educational', 'Ride haling', 'Finance_Invest...",4,0,1,0,0,1,1,1
1,573f28fe9b0ffc28367719f1,"['zoom', 'linkedin', 'microsoft teams', 'netfl...",25,"['Tools', 'Social', 'News', 'OTT', 'Delivery_G...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1
2,573f29089b0ffc2836773ef0,"['telegram', 'zoom', 'cred', 'zomato', 'messen...",30,"['Tools', 'Social', 'News', 'Educational', 'De...",12,"['Educational', 'Finance_Investment', 'Office'...",5,0,1,0,1,1,1,1
3,573f290e9b0ffc2836775ef5,"['telegram', 'zomato', 'linkedin', 'indigo', '...",23,"['Tools', 'Fantasy_Sports', 'Social', 'Deliver...",12,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1
4,573f292a9b0ffc283677b413,"['instagram', 'facebook', 'kite by zerodha', '...",6,"['OTT', 'Social', 'Finance_Investment', 'Deliv...",4,"['Finance_Investment', 'Rest']",2,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130486,6609a39da9381bdd0b3a1ec6,"['instagram', 'disney+ hotstar', 'jio cinema']",3,"['Social', 'OTT']",2,['Rest'],1,0,0,0,0,0,0,1
130487,6609a3f78fcbab6f75021b56,"['telegram', 'instagram', 'google news', 'face...",13,"['Tools', 'Social', 'News', 'Delivery_Food', '...",8,"['Ride haling', 'Rest']",2,0,0,0,0,0,1,1
130488,6609a49e8fcbabed6c023a50,"['instagram', 'facebook', 'flipkart', 'truecal...",10,"['Tools', 'Social', 'OTT', 'Finance_Transactio...",5,['Rest'],1,0,0,0,0,0,0,1
130489,6609a552db0e9c01003fa360,"['instagram', 'facebook', 'truecaller', 'linke...",11,"['Tools', 'Social', 'OTT', 'Streaming_Music', ...",6,"['Finance_Investment', 'Rest']",2,0,0,0,0,1,0,1


In [20]:
df_customer_data = pd.merge(df_bangalore_active_customer, df_customer_mapped,
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(130491, 29)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,5e3b93c0d6d3936cd767fbdb,MALE,255.0,30.0,6.0,3.0,2.0,PHH,14,HOOK,HIGH_INCOME,ONLY_AUTO,LONG_DISTANCE,NPS,2.0,c. medium rpc,"['telegram', 'ludo king', 'instagram', 'cred',...",31,"['Tools', 'Social', 'News', 'Educational', 'De...",11,"['Educational', 'Gaming', 'Finance_Investment'...",6,0,1,1,1,1,1,1


#### Derived columns

In [21]:
## RPC

df_customer_data['rpc'] =  df_customer_data['net_count'].replace(0, np.nan)
df_customer_data.rpc.describe()

count    111437.000000
mean          2.456715
std           2.979753
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max          62.000000
Name: rpc, dtype: float64

In [22]:
df_customer_data['rpc_bucket'] = np.where(df_customer_data['net_count'] < 1 , 'a. ZERO',
                                    np.where(df_customer_data['net_count'] < 2 , 'b. LOW',
                                        np.where(df_customer_data['net_count'] < 4 , 'c. MEDIUM', 
                                            np.where(df_customer_data['net_count'] >= 4 , 'd. HIGH', None))))

In [23]:
## app_count_bucket

df_customer_data.app_count.describe()

count    130491.000000
mean         15.790622
std           8.032980
min           1.000000
25%          10.000000
50%          15.000000
75%          21.000000
max          73.000000
Name: app_count, dtype: float64

In [24]:
df_customer_data['app_count_bucket'] = np.where(df_customer_data['net_count'] < 5 , '1-5',
                                        np.where(df_customer_data['net_count'] < 10 , '6-10',
                                          np.where(df_customer_data['net_count'] < 16 , '11-15', 'Above 15')))

In [25]:
## category_count_bucket

df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.category_count.describe()

count    130491.000000
mean          2.709988
std           1.258384
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

In [26]:
df_customer_data['category_count_bucket'] = np.where(df_customer_data['category_count'] < 2 , '1',
                                              np.where(df_customer_data['category_count'] < 3 , '2',
                                                np.where(df_customer_data['category_count'] < 4 , '3','Above 3')))

In [27]:
## category_count

# df_customer_data['category_count'] =  df_customer_data['categories_l2_count'].replace(0, 1)
df_customer_data.rapido_age.describe()

count    130479.000000
mean        680.803808
std         573.034119
min           2.000000
25%         232.000000
50%         559.000000
75%         899.000000
max        2875.000000
Name: rapido_age, dtype: float64

#### Merge

In [28]:
df_customer_data_explode = pd.merge(df_customer_data,
                                    df_customer_cat_mapping,
                                    how='inner', on =['customer_id'])

df_customer_data_explode[df_customer_data_explode['customer_id'] == '6475f56aba9a99b915ccc086'].head()

,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc,app_list,app_count,categories_l1,categories_l1_count,categories_l2_x,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest,rpc_bucket,app_count_bucket,category_count,category_count_bucket,categories_l2_y


## User Base Distribution analysis

LTR-Segment

Service Affinity

Distance Affinity

Customers Use Case

AO- Net Funnel

Gender

Age (Whatever fill rate we have)

Income

RPC

In [29]:
tot_customers = df_customer_cat_mapping.customer_id.nunique()
df_category_agg = df_customer_cat_mapping\
                    .groupby(['categories_l2'])\
                    .agg(total_customers = ('customer_id','nunique'))\
                    .sort_values(['total_customers'], ascending=False)\
                    .reset_index()
df_category_agg['% distribution'] =  (df_category_agg['total_customers']*100.00/tot_customers).round(2)
df_category_agg

,categories_l2,total_customers,% distribution
0,Rest,130450,99.97
1,Office,60031,46.00
2,Ride haling,53987,41.37
3,Finance_Investment,45953,35.22
4,Educational,29021,22.24
5,Gaming,25116,19.25
6,Driver_App,9071,6.95


#### LTR-Segment

In [30]:
ltr_segment_city = df_customer_data\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ltr_segment_city['% city threshold'] =  (ltr_segment_city['customers']*100.00/ltr_segment_city.customers.sum()).round(2)
ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,28364,21.74
1,LTR 0,3560,2.73
2,PHH,98491,75.48
3,UNKNOWN,76,0.06


In [31]:
df1 = df_customer_data_explode[~df_customer_data_explode['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers'])

customers                                                      \
categories_l2 Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                    
HH                  2429        5402               8025   5061  10627  28352   
LTR 0                370         618                831    682   1136   3555   
PHH                 6268       22977              37067  19358  48223  98467   

                           
categories_l2 Ride haling  
ltr_segment                
HH                   8379  
LTR 0                 820  
PHH                 44738

In [32]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customer %'])

In [33]:
print('% Distribution of customers across individual Segment.')
df2.pivot(index ='ltr_segment' , columns ='categories_l2', values =['customers %'])

% Distribution of customers across individual Segment.


customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
ltr_segment                                                                     
HH                   8.56       19.05              28.29  17.84  37.47  99.96   
LTR 0               10.39       17.36              23.34  19.16  31.91  99.86   
PHH                  6.36       23.33              37.63  19.65  48.96  99.98   

                           
categories_l2 Ride haling  
ltr_segment                
HH                  29.54  
LTR 0               23.03  
PHH                 45.42

#### Other testing

In [34]:
df_test = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l2'].isin(['Office', 
                                                                                         'Educational',
                                                                                         'Ride haling'
                                                                                        ])]
df_test = pd.merge(df_test,df_bangalore_active_customer[['customer_id', 'ltr_segment']], how='inner', on='customer_id')
df_office = df_test[df_test['categories_l2'].isin(['Office'])]
df_education = df_test[df_test['categories_l2'].isin(['Educational'])]
df_ride_hailing = df_test[df_test['categories_l2'].isin(['Ride haling'])]

##### Explode - Ride Hailing

In [35]:
rh_ltr_segment_city = df_ride_hailing\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rh_ltr_segment_city['% city threshold'] =  (rh_ltr_segment_city['customers']*100.00/rh_ltr_segment_city.customers.sum()).round(2)
rh_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,8379,15.52
1,LTR 0,820,1.52
2,PHH,44738,82.87
3,UNKNOWN,50,0.09


In [36]:
df1 = df_ride_hailing[~df_ride_hailing['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
ride_hailing_total_customers = df_ride_hailing.customer_id.nunique()
df2 = pd.merge(df1,rh_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/ride_hailing_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                  \
app_name     blusmart driveu driver app in drive   jugnoo namma yatri   
ltr_segment                                                             
HH               27.0               1.0    281.0   1478.0        14.0   
LTR 0             NaN               NaN     42.0    136.0         NaN   
PHH             197.0               6.0   1711.0  10256.0       142.0   

                                                    
app_name         ola quick ride quickride     uber  
ltr_segment                                         
HH            5784.0        6.0       NaN   3504.0  
LTR 0          585.0        1.0       NaN    332.0  
PHH          33769.0       71.0       1.0  19145.0

In [37]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [38]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                       \
app_name       blusmart driveu driver app in drive jugnoo namma yatri    ola   
ltr_segment                                                                    
HH                 0.32              0.01     3.35  17.64        0.17  69.03   
LTR 0               NaN               NaN     5.12  16.59         NaN  71.34   
PHH                0.44              0.01     3.82  22.92        0.32  75.48   

                                         
app_name    quick ride quickride   uber  
ltr_segment                              
HH                0.07       NaN  41.82  
LTR 0             0.12       NaN  40.49  
PHH               0.16       0.0  42.79

##### Explode - Office

In [39]:
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
0,5e3b93c0d6d3936cd767fbdb,intune company portal,intune company portal,Office,Office,PHH
1,5e3b93c0d6d3936cd767fbdb,outlook,outlook,Office,Office,PHH


In [40]:
def assign_office_value(row):
    if row['app_name'] in ['asana','cisco','confluence','github','google analytics',
                           'intune  company portal','jira','miro','slack','trello','zoho mail','zoho meeting']:
        return 'code_office_app'
    else:
        return 'secondary_office_app'

df_office['app_name_tag'] = df_office.apply(assign_office_value, axis=1)
df_office.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
0,5e3b93c0d6d3936cd767fbdb,intune company portal,intune company portal,Office,Office,PHH,code_office_app
1,5e3b93c0d6d3936cd767fbdb,outlook,outlook,Office,Office,PHH,secondary_office_app


In [41]:
df_office[df_office['app_name_tag'] == 'code_office_app'].app_name.unique()

array(['intune  company portal', 'trello', 'zoho mail', 'slack', 'github',
       'google analytics', 'jira', 'cisco', 'asana', 'zoho meeting',
       'confluence', 'miro'], dtype=object)

In [42]:
office_ltr_segment_city = df_office\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
office_ltr_segment_city['% city threshold'] =  (office_ltr_segment_city['customers']*100.00/office_ltr_segment_city.customers.sum()).round(2)
office_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,10627,17.70
1,LTR 0,1136,1.89
2,PHH,48223,80.33
3,UNKNOWN,45,0.07


In [43]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                   \
app_name        asana cisco confluence dropbox github google analytics   
ltr_segment                                                              
HH               10.0   3.0        3.0    94.0   94.0             19.0   
LTR 0             1.0   NaN        NaN     8.0   16.0              1.0   
PHH              75.0  25.0       18.0   537.0  647.0            101.0   

                                                                              \
app_name    google authenticator intune  company portal   jira microsoft 365   
ltr_segment                                                                    
HH                         654.0                  610.0   17.0        3449.0   
LTR 0                       64.0                   44.0    3.0         340.0   
PHH                       4241.0                 4143.0  177.0       16201.0   

                                                                             \
app_name    microsoft teams  miro ms authenticator onedrive  outlook pocket   
ltr_segment                                                                   
HH                   2333.0   3.0            821.0   4833.0   2964.0    2.0   
LTR 0                 187.0   NaN             50.0    549.0    309.0    NaN   
PHH                 13878.0  18.0           5625.0  19262.0  14755.0   62.0   

                                                                    
app_name      slack trello   webex zoho mail zoho meeting     zoom  
ltr_segment                                                         
HH            179.0   20.0   438.0      58.0         22.0   4581.0  
LTR 0          12.0    1.0    36.0       6.0          2.0    488.0  
PHH          1572.0  138.0  2358.0     412.0        134.0  22492.0

In [44]:
# print('% Distribution of customers across individual categories.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [45]:
print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual app.


customers %                                                   \
app_name          asana cisco confluence dropbox github google analytics   
ltr_segment                                                                
HH                 0.09  0.03       0.03    0.88   0.88             0.18   
LTR 0              0.09   NaN        NaN    0.70   1.41             0.09   
PHH                0.16  0.05       0.04    1.11   1.34             0.21   

                                                                             \
app_name    google authenticator intune  company portal  jira microsoft 365   
ltr_segment                                                                   
HH                          6.15                   5.74  0.16         32.46   
LTR 0                       5.63                   3.87  0.26         29.93   
PHH                         8.79                   8.59  0.37         33.60   

                                                                            \
app_name    microsoft teams  miro ms authenticator onedrive outlook pocket   
ltr_segment                                                                  
HH                    21.95  0.03             7.73    45.48   27.89   0.02   
LTR 0                 16.46   NaN             4.40    48.33   27.20    NaN   
PHH                   28.78  0.04            11.66    39.94   30.60   0.13   

                                                              
app_name    slack trello webex zoho mail zoho meeting   zoom  
ltr_segment                                                   
HH           1.68   0.19  4.12      0.55         0.21  43.11  
LTR 0        1.06   0.09  3.17      0.53         0.18  42.96  
PHH          3.26   0.29  4.89      0.85         0.28  46.64

In [46]:
df1 = df_office[~df_office['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
office_total_customers = df_office.customer_id.nunique()
df2 = pd.merge(df1,office_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/office_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                       963                10521
LTR 0                     80                 1129
PHH                     6802                47513

In [47]:
#### print('% Distribution of customers across individual app.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

customers %                     
app_name_tag code_office_app secondary_office_app
ltr_segment                                      
HH                      9.06                99.00
LTR 0                   7.04                99.38
PHH                    14.11                98.53

##### Explode - Education

In [48]:
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment
5,5e3b93c0d6d3936cd767fbdb,duolingo,duolingo,Educational,Educational,PHH
8,612f6a009db16b0477ee6780,udemy,udemy,Educational,Educational,PHH


In [49]:
def assign_education_value(row):
    if row['app_name'] in ['aakash','byju','chegg study','diksha','e-pg pathshala',
                           'google classroom', 'microsoft math solver','moodle','mycbseguide', 
                           'physics wallah','simplilearn','vedantu','vidyakul'
                          ]:
#         brainly duolingo
        return 'code_education_app'
    else:
        return 'secondary_education_app'

df_education['app_name_tag'] = df_education.apply(assign_education_value, axis=1)
df_education.head(2)

,customer_id,app_list_set,app_name,categories_l2,categories_l1,ltr_segment,app_name_tag
5,5e3b93c0d6d3936cd767fbdb,duolingo,duolingo,Educational,Educational,PHH,secondary_education_app
8,612f6a009db16b0477ee6780,udemy,udemy,Educational,Educational,PHH,secondary_education_app


In [50]:
df_education[df_education['app_name_tag'] == 'code_education_app'].app_name.unique()

array(['google classroom', 'physics wallah', 'microsoft math solver',
       'simplilearn', 'byju', 'mycbseguide', 'diksha', 'e-pg pathshala',
       'aakash', 'moodle', 'vedantu', 'chegg study', 'vidyakul'],
      dtype=object)

In [51]:
edu_ltr_segment_city = df_education\
                        .groupby(['ltr_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
edu_ltr_segment_city['% city threshold'] =  (edu_ltr_segment_city['customers']*100.00/edu_ltr_segment_city.customers.sum()).round(2)
edu_ltr_segment_city

,ltr_segment,customers,% city threshold
0,HH,5402,18.61
1,LTR 0,618,2.13
2,PHH,22977,79.17
3,UNKNOWN,24,0.08


In [52]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers'])

customers                                                       \
app_name       aakash brainly    byju caclubindia chegg study colegeduniya   
ltr_segment                                                                  
HH               38.0   582.0   553.0         NaN         9.0          7.0   
LTR 0             5.0    56.0    80.0         NaN         1.0          NaN   
PHH             103.0  2382.0  1520.0         7.0        49.0         28.0   

                                                                        \
app_name    collegedekho course hero coursera diksha doubtnut duolingo   
ltr_segment                                                              
HH                   1.0         NaN    157.0  108.0    412.0    898.0   
LTR 0                NaN         NaN     13.0   10.0     58.0    109.0   
PHH                  4.0         3.0   1265.0  282.0   1248.0   3683.0   

                                                                    \
app_name    e-pg pathshala    edx embibe geeks for geeks goodreads   
ltr_segment                                                          
HH                    67.0   31.0   32.0            59.0      10.0   
LTR 0                  4.0    2.0    2.0             6.0       2.0   
PHH                  292.0  180.0  138.0           391.0     121.0   

                                                           \
app_name    google classroom ignou e-content khan academy   
ltr_segment                                                 
HH                    1362.0            17.0         15.0   
LTR 0                  129.0             1.0          NaN   
PHH                   6205.0           110.0         68.0   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                            6.0   28.0           2.0        58.0   
LTR 0                         1.0    3.0           NaN         8.0   
PHH                          65.0  118.0           6.0       189.0   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                116.0      25.0         1110.0    2.0         132.0   
LTR 0               8.0       8.0          128.0    NaN          17.0   
PHH               603.0     119.0         4395.0   62.0         355.0   

                                                                          \
app_name    shiksha.com simplilearn stack overflow  swayam toppr   udemy   
ltr_segment                                                                
HH                  9.0        52.0            NaN   211.0   3.0   409.0   
LTR 0               1.0         6.0            NaN    21.0   NaN    29.0   
PHH                47.0       326.0            5.0  1045.0   NaN  2965.0   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH              794.0    21.0      1.0        10.0  
LTR 0            93.0     2.0      NaN         1.0  
PHH            3973.0    81.0      2.0        35.0

In [53]:
# print('% Distribution of customers across individual apps.')
# df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customer %'])

In [54]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name', values =['customers %'])

% Distribution of customers across individual segment.


customers %                                                      \
app_name         aakash brainly   byju caclubindia chegg study colegeduniya   
ltr_segment                                                                   
HH                 0.70   10.77  10.24         NaN        0.17         0.13   
LTR 0              0.81    9.06  12.94         NaN        0.16          NaN   
PHH                0.45   10.37   6.62        0.03        0.21         0.12   

                                                                        \
app_name    collegedekho course hero coursera diksha doubtnut duolingo   
ltr_segment                                                              
HH                  0.02         NaN     2.91   2.00     7.63    16.62   
LTR 0                NaN         NaN     2.10   1.62     9.39    17.64   
PHH                 0.02        0.01     5.51   1.23     5.43    16.03   

                                                                   \
app_name    e-pg pathshala   edx embibe geeks for geeks goodreads   
ltr_segment                                                         
HH                    1.24  0.57   0.59            1.09      0.19   
LTR 0                 0.65  0.32   0.32            0.97      0.32   
PHH                   1.27  0.78   0.60            1.70      0.53   

                                                           \
app_name    google classroom ignou e-content khan academy   
ltr_segment                                                 
HH                     25.21            0.31         0.28   
LTR 0                  20.87            0.16          NaN   
PHH                    27.01            0.48         0.30   

                                                                    \
app_name    microsoft math solver moodle my study life mycbseguide   
ltr_segment                                                          
HH                           0.11   0.52          0.04        1.07   
LTR 0                        0.16   0.49           NaN        1.29   
PHH                          0.28   0.51          0.03        0.82   

                                                                       \
app_name    ncert books photomath physics wallah pocket shiksha mitra   
ltr_segment                                                             
HH                 2.15      0.46          20.55   0.04          2.44   
LTR 0              1.29      1.29          20.71    NaN          2.75   
PHH                2.62      0.52          19.13   0.27          1.55   

                                                                        \
app_name    shiksha.com simplilearn stack overflow swayam toppr  udemy   
ltr_segment                                                              
HH                 0.17        0.96            NaN   3.91  0.06   7.57   
LTR 0              0.16        0.97            NaN   3.40   NaN   4.69   
PHH                0.20        1.42           0.02   4.55   NaN  12.90   

                                                    
app_name    unacademy vedantu vidyakul whitehat jr  
ltr_segment                                         
HH              14.70    0.39     0.02        0.19  
LTR 0           15.05    0.32      NaN        0.16  
PHH             17.29    0.35     0.01        0.15

In [55]:
df1 = df_education[~df_education['ltr_segment'].isin(['UNKNOWN'])]\
.groupby(['app_name_tag', 'ltr_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['app_name_tag', 'customers'],ascending=[False,True])\
.reset_index()
edu_total_customers = df_education.customer_id.nunique()
df2 = pd.merge(df1,edu_ltr_segment_city[['ltr_segment','customers']], how= 'left', on='ltr_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/edu_total_customers).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers'])

customers                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                         3126                    3250
LTR 0                       351                     362
PHH                       12470                   14901

In [56]:
print('% Distribution of customers across individual segment.')
df2.pivot(index ='ltr_segment' , columns ='app_name_tag', values =['customers %'])

% Distribution of customers across individual segment.


customers %                        
app_name_tag code_education_app secondary_education_app
ltr_segment                                            
HH                        57.87                   60.16
LTR 0                     56.80                   58.58
PHH                       54.27                   64.85

### Other

#### Service Affinity

In [57]:
service_affinity_city = df_customer_data\
                        .groupby(['service_affinity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
service_affinity_city['% city threshold']=(service_affinity_city['customers']*100.00/service_affinity_city.customers.sum()).round(2)
service_affinity_city.sort_values(by=['service_affinity'],ascending=[True])

,service_affinity,customers,% city threshold
0,AUTO_CAB,648,0.50
1,LINK_AUTO,11568,8.86
2,LINK_CAB,691,0.53
3,NO_AFFINITY,4302,3.30
4,ONLY_AUTO,19307,14.80
5,ONLY_CAB,2173,1.67
6,ONLY_LINK,84844,65.02
7,UNKNOWN,6958,5.33


In [58]:
df1 = df_customer_data_explode[~df_customer_data_explode['service_affinity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'service_affinity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,service_affinity_city[['service_affinity','customers']], how= 'left', on='service_affinity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='service_affinity' , columns ='categories_l2', values =['customers %', 'customers'])

customers %                                               \
categories_l2     Driver_App Educational Finance_Investment Gaming Office   
service_affinity                                                            
AUTO_CAB                4.78       23.77              40.43  21.14  56.48   
LINK_AUTO               5.63       24.62              34.05  19.53  45.91   
LINK_CAB               11.87       19.10              40.67  23.88  48.19   
NO_AFFINITY             7.32       22.22              38.82  20.04  48.72   
ONLY_AUTO               4.54       26.10              30.29  18.91  47.61   
ONLY_CAB               22.87       18.22              31.06  14.54  40.13   
ONLY_LINK               7.14       21.23              36.48  19.31  45.79   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
service_affinity                                              
AUTO_CAB          100.00       54.94       31.0       154.0   
LINK_AUTO          99.96       42.69      651.0      2848.0   
LINK_CAB           99.86       50.07       82.0       132.0   
NO_AFFINITY        99.95       53.09      315.0       956.0   
ONLY_AUTO         100.00       44.96      876.0      5039.0   
ONLY_CAB          100.00       58.08      497.0       396.0   
ONLY_LINK          99.97       39.74     6062.0     18010.0   

                                                                            
categories_l2    Finance_Investment   Gaming   Office     Rest Ride haling  
service_affinity                                                            
AUTO_CAB                      262.0    137.0    366.0    648.0       356.0  
LINK_AUTO                    3939.0   2259.0   5311.0  11563.0      4938.0  
LINK_CAB                      281.0    165.0    333.0    690.0       346.0  
NO_AFFINITY                  1670.0    862.0   2096.0   4300.0      2284.0  
ONLY_AUTO                    5848.0   3650.0   9192.0  19307.0      8681.0  
ONLY_CAB                      675.0    316.0    872.0   2173.0      1262.0  
ONLY_LINK                   30955.0  16386.0  38853.0  84816.0     33718.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be LINK Customers.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be AUTO Customers.
    - Educational, Office, Ride haling

#### Distance Preference

In [59]:
distance_affinity_city = df_customer_data\
                        .groupby(['distance_preference'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
distance_affinity_city['% city threshold']=(distance_affinity_city['customers']*100.00/distance_affinity_city.customers.sum()).round(2)
distance_affinity_city.sort_values(by=['distance_preference'],ascending=[True])

,distance_preference,customers,% city threshold
0,LONG_DISTANCE,34324,26.30
1,MEDIUM_DISTANCE,31718,24.31
2,SHORT_DISTANCE,29467,22.58
3,UNKNOWN,34982,26.81


In [60]:
df1 = df_customer_data_explode[~df_customer_data_explode['distance_preference'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'distance_preference'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,distance_affinity_city[['distance_preference','customers']], how= 'left', on='distance_preference')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='distance_preference' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2        Driver_App Educational Finance_Investment Gaming Office   
distance_preference                                                            
LONG_DISTANCE              8.19       20.08              38.18  18.65  47.99   
MEDIUM_DISTANCE            6.32       21.93              34.63  18.40  44.39   
SHORT_DISTANCE             5.45       24.90              32.41  19.70  45.81   

                                        customers              \
categories_l2         Rest Ride haling Driver_App Educational   
distance_preference                                             
LONG_DISTANCE        99.95       43.37     2811.0      6891.0   
MEDIUM_DISTANCE      99.97       39.65     2006.0      6955.0   
SHORT_DISTANCE       99.98       41.43     1606.0      7337.0   

                                                                              
categories_l2       Finance_Investment  Gaming   Office     Rest Ride haling  
distance_preference                                                           
LONG_DISTANCE                  13104.0  6400.0  16471.0  34306.0     14886.0  
MEDIUM_DISTANCE                10983.0  5837.0  14081.0  31710.0     12576.0  
SHORT_DISTANCE                  9549.0  5804.0  13499.0  29461.0     12207.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to do LONG Distance rides.
    - Driver_App, Gaming
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do MEDIUM Distance rides.
    - Finance_Investment
   
- Whenever an app belongs to the app-categories listed below, customers are more likely to do SHORT Distance rides.
    - Educational, Office

#### Gender

In [61]:
gender_city = df_customer_data\
                        .groupby(['gender'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
gender_city['% city threshold']=(gender_city['customers']*100.00/gender_city.customers.sum()).round(2)
gender_city.sort_values(by=['gender'],ascending=[True])

,gender,customers,% city threshold
0,FEMALE,32851,25.17
1,MALE,93688,71.80
2,OTHERS,229,0.18
3,UNKNOWN,3723,2.85


In [62]:
df1 = df_customer_data_explode[~df_customer_data_explode['gender'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'gender'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,gender_city[['gender','customers']], how= 'left', on='gender')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='gender' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   
gender                                                                   
FEMALE               2.24       27.46              21.44  21.40  46.23   
MALE                 8.77       20.29              40.26  18.44  45.99   
OTHERS               6.11       15.28              15.28  23.58  30.13   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
gender                                                                        
FEMALE          99.98       44.81      737.0      9020.0             7042.0   
MALE            99.96       40.21     8213.0     19008.0            37715.0   
OTHERS         100.00       27.95       14.0        35.0               35.0   

                                                      
categories_l2   Gaming   Office     Rest Ride haling  
gender                                                
FEMALE          7029.0  15188.0  32844.0     14720.0  
MALE           17274.0  43086.0  93654.0     37673.0  
OTHERS            54.0     69.0    229.0        64.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be MALE.
    - Driver_App, Finance_Investment
- Whenever an app belongs to the app-categories listed below, customers are more likely to be FEMALE.
    - Educational

#### Income Segment

In [63]:
income_segment_city = df_customer_data\
                        .groupby(['income_segment'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
income_segment_city['% city threshold']=(income_segment_city['customers']*100.00/income_segment_city.customers.sum()).round(2)
income_segment_city.sort_values(by=['income_segment'],ascending=[True])

,income_segment,customers,% city threshold
0,HIGH_INCOME,46097,35.33
1,LOW_INCOME,8651,6.63
2,MEDIUM_INCOME,56199,43.07
3,UNKNOWN,19544,14.98


In [64]:
df1 = df_customer_data_explode[~df_customer_data_explode['income_segment'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'income_segment'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,income_segment_city[['income_segment','customers']], how= 'left', on='income_segment')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='income_segment' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2   Driver_App Educational Finance_Investment Gaming Office   
income_segment                                                            
HIGH_INCOME           7.64       23.96              43.83  22.50  53.71   
LOW_INCOME            5.09       15.89              18.96  10.48  27.71   
MEDIUM_INCOME         6.56       23.63              31.16  18.42  44.85   

                                   customers                                 \
categories_l2    Rest Ride haling Driver_App Educational Finance_Investment   
income_segment                                                                
HIGH_INCOME     99.99       52.05     3522.0     11047.0            20203.0   
LOW_INCOME      99.91       21.27      440.0      1375.0             1640.0   
MEDIUM_INCOME   99.96       37.76     3688.0     13278.0            17513.0   

                                                       
categories_l2    Gaming   Office     Rest Ride haling  
income_segment                                         
HIGH_INCOME     10373.0  24760.0  46092.0     23992.0  
LOW_INCOME        907.0   2397.0   8643.0      1840.0  
MEDIUM_INCOME   10353.0  25204.0  56177.0     21221.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be HIGH_INCOME.
    - Office, Educational, Finance_Investment, Gaming, Ride haling
- Whenever an app belongs to the app-categories listed below, customers are more likely to be MEDIUM_INCOME.
    - Driver_App

#### Price Sensitivity

In [65]:
ps_city = df_customer_data\
                        .groupby(['price_sensitivity'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
ps_city['% city threshold']=(ps_city['customers']*100.00/ps_city.customers.sum()).round(2)
ps_city.sort_values(by=['price_sensitivity'],ascending=[True])

,price_sensitivity,customers,% city threshold
0,NPS,31453,24.10
1,PS,26244,20.11
2,UNKNOWN,72794,55.78


In [66]:
df1 = df_customer_data_explode[~df_customer_data_explode['price_sensitivity'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'price_sensitivity'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,ps_city[['price_sensitivity','customers']], how= 'left', on='price_sensitivity')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='price_sensitivity' , columns ='categories_l2', values =['customers %','customers'])

customers %                                               \
categories_l2      Driver_App Educational Finance_Investment Gaming Office   
price_sensitivity                                                            
NPS                      6.92       21.85              34.93  20.69  46.95   
PS                       6.19       23.96              35.03  20.52  47.33   

                                      customers              \
categories_l2       Rest Ride haling Driver_App Educational   
price_sensitivity                                             
NPS                99.99       44.01     2177.0      6872.0   
PS                 99.97       45.93     1624.0      6288.0   

                                                                            
categories_l2     Finance_Investment  Gaming   Office     Rest Ride haling  
price_sensitivity                                                           
NPS                          10986.0  6507.0  14767.0  31450.0     13844.0  
PS                            9193.0  5385.0  12421.0  26236.0     12054.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers are more likely to be Price Sensitivity.
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers are less likely to be Price Sensitivity. (Less confidence - Uninterpretable/Hard to interpretable using Price Sensitivity)
    - Office, Educational, Finance_Investment, Ride haling, Gaming

#### RPC

In [67]:
rpc_bucket_city = df_customer_data\
                        .groupby(['rpc_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
rpc_bucket_city['% city threshold']=(rpc_bucket_city['customers']*100.00/rpc_bucket_city.customers.sum()).round(2)
rpc_bucket_city.sort_values(by=['rpc_bucket'],ascending=[True])

,rpc_bucket,customers,% city threshold
0,a. ZERO,19033,14.59
1,b. LOW,59473,45.58
2,c. MEDIUM,33336,25.55
3,d. HIGH,18628,14.28


In [68]:
df1 = df_customer_data_explode[~df_customer_data_explode['rpc_bucket'].isin(['UNKNOWN'])]\
.groupby(['categories_l2_y', 'rpc_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'customers'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)
df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2 = pd.merge(df2,rpc_bucket_city[['rpc_bucket','customers']], how= 'left', on='rpc_bucket')
df2.rename(columns = {'customers_x' :'customers',
                      'customers_y' : 'segment_wise_customers'
                     }, inplace = True)
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2['customers %'] =  (df2['customers']*100.00/df2['segment_wise_customers']).round(2)
df2.pivot(index ='rpc_bucket' , columns ='categories_l2', values =['customers %','customers'])

customers %                                                      \
categories_l2  Driver_App Educational Finance_Investment Gaming Office   Rest   
rpc_bucket                                                                      
a. ZERO              9.76       21.42              33.32  20.08  42.18  99.96   
b. LOW               7.11       22.41              35.30  18.80  45.30  99.97   
c. MEDIUM            6.14       22.60              36.10  19.47  47.33  99.97   
d. HIGH              5.02       21.91              35.30  19.44  49.79  99.96   

                           customers                                          \
categories_l2 Ride haling Driver_App Educational Finance_Investment   Gaming   
rpc_bucket                                                                     
a. ZERO             37.03     1857.0      4076.0             6341.0   3822.0   
b. LOW              39.72     4229.0     13326.0            20994.0  11179.0   
c. MEDIUM           42.64     2047.0      7535.0            12033.0   6491.0   
d. HIGH             48.84      935.0      4081.0             6575.0   3622.0   

                                             
categories_l2   Office     Rest Ride haling  
rpc_bucket                                   
a. ZERO         8028.0  19026.0      7047.0  
b. LOW         26943.0  59456.0     23620.0  
c. MEDIUM      15778.0  33326.0     14215.0  
d. HIGH         9274.0  18621.0      9097.0

Insights 

- Whenever an app belongs to the app-categories listed below, customers having Less RPC
    - Driver_App
- Whenever an app belongs to the app-categories listed below, customers having Good RPC
    - Educational, Finance_Investment, Office, Ride haling

#### App Count Bucket

In [69]:
app_count_bucket_city = df_customer_data\
                        .groupby(['app_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'))\
                        .reset_index()
app_count_bucket_city['% city threshold']=(app_count_bucket_city['customers']*100.00/app_count_bucket_city.customers.sum()).round(2)
app_count_bucket_city.sort_values(by=['app_count_bucket'],ascending=[True])

,app_count_bucket,customers,% city threshold
0,1-5,117410,89.98
1,11-15,2643,2.03
2,6-10,9155,7.02
3,Above 15,1283,0.98


In [70]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'app_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'app_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='app_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2    Driver_App Educational Finance_Investment Gaming Office   
app_count_bucket                                                           
1-5                   93.00       90.35              90.05  89.75  88.99   
11-15                  1.36        1.75               2.05   2.13   2.36   
6-10                   4.93        7.10               6.92   7.20   7.56   
Above 15               0.72        0.80               0.98   0.92   1.09   

                                     
categories_l2      Rest Ride haling  
app_count_bucket                     
1-5               89.97       87.90  
11-15              2.03        2.56  
6-10               7.02        8.24  
Above 15           0.98        1.30

In [71]:
df_customer_data.app_count.describe()

count    130491.000000
mean         15.790622
std           8.032980
min           1.000000
25%          10.000000
50%          15.000000
75%          21.000000
max          73.000000
Name: app_count, dtype: float64

#### Category Count Bucket

In [72]:
cat_count_bucket_city = df_customer_data\
                        .groupby(['category_count_bucket'])\
                        .agg(customers = ('customer_id','nunique'),)\
                        .reset_index()
cat_count_bucket_city['% city threshold']=(cat_count_bucket_city['customers']*100.00/cat_count_bucket_city.customers.sum()).round(2)
cat_count_bucket_city.sort_values(by=['category_count_bucket'],ascending=[True])

,category_count_bucket,customers,% city threshold
0,1,25317,19.40
1,2,35990,27.58
2,3,34032,26.08
3,Above 3,35152,26.94


In [73]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y', 'category_count_bucket'])\
.agg(customers = ('customer_id','nunique'))\
.sort_values(by=['categories_l2_y', 'category_count_bucket'],ascending=[False,True])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['customer %'] =  (df2['customers']*100.00/df2['total_customers']).round(2)
df2.pivot(index ='category_count_bucket' , columns ='categories_l2', values =['customer %'])

customer %                                               \
categories_l2         Driver_App Educational Finance_Investment Gaming Office   
category_count_bucket                                                           
1                           0.01        0.01               0.02   0.02   0.01   
2                          18.61       15.42              16.63  17.61  17.00   
3                          32.74       26.78              29.52  27.68  32.80   
Above 3                    48.64       57.79              53.84  54.70  50.18   

                                          
categories_l2           Rest Ride haling  
category_count_bucket                     
1                      19.39        0.01  
2                      27.58       14.03  
3                      26.09       31.70  
Above 3                26.95       54.27

In [74]:
df_temp = df_customer_data[df_customer_data['category_count_bucket'].isin(['2-3'])]\
.groupby(['customer_id'])\
.agg({'categories_l2' : set})\
.reset_index()
df_temp[['categories_l2']]

,categories_l2


In [75]:
df_customer_data.category_count.describe()

count    130491.000000
mean          2.709988
std           1.258384
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           7.000000
Name: category_count, dtype: float64

#### AO-NET Funnel

In [76]:
df_customer_data['city'] = 'Jaipur, Android'
city_funnel = df_customer_data\
                        .groupby(['city'])\
                        .agg(fe_count = ('fe_count','sum'),
                             rr_count = ('rr_count','sum'),
                             net_count = ('net_count','sum')
                            )\
                        .reset_index()

city_funnel['% fe2rr']=(city_funnel['rr_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel['% g2n']=(city_funnel['net_count']*100.00/city_funnel['rr_count']).round(2)
city_funnel['% fe2net']=(city_funnel['net_count']*100.00/city_funnel['fe_count']).round(2)
city_funnel[['city','% fe2rr','% g2n','% fe2net']].T

,0
city,"Jaipur, Android"
% fe2rr,43.86
% g2n,67.04
% fe2net,29.41


In [78]:
df1 = df_customer_data_explode\
.groupby(['categories_l2_y'])\
.agg(customers = ('customer_id','nunique'),
     fe_count = ('fe_count','sum'),
     rr_count = ('rr_count','sum'),
     net_count = ('net_count','sum')
    )\
.sort_values(by=['categories_l2_y'],ascending=[False])\
.reset_index()

df1.rename(columns = {'categories_l2_y' :'categories_l2'}, inplace = True)

df2 = pd.merge(df_category_agg,df1, how= 'inner', on='categories_l2')
df2['% fe2rr']=(df2['rr_count']*100.00/df2['fe_count']).round(2)
df2['% g2n']=(df2['net_count']*100.00/df2['rr_count']).round(2)
df2['% fe2net']=(df2['net_count']*100.00/df2['fe_count']).round(2)
df2[['categories_l2','% fe2rr','% g2n','% fe2net']].T

,0,1,2,3,4,5,6
categories_l2,Rest,Office,Ride haling,Finance_Investment,Educational,Gaming,Driver_App
% fe2rr,43.87,44.91,45.89,43.62,42.71,44.89,43.47
% g2n,67.04,69.51,69.1,69.01,67.11,65.53,58.84
% fe2net,29.41,31.21,31.7,30.1,28.66,29.42,25.58
